In [1]:
from multiprocessing import Process, Manager, Event
from threading import Thread
import pandas as pd
import traceback
import websocket
import json
import numpy as np
import datetime
import time
# set directory to upper directory
import os
import sys
# sys.path.append(os.path.dirname(os.path.abspath(os.path.dirname(__file__))))
sys.path.append("/home/kp_info_loader") # TEMP
from loggers.logger import KimpBotLogger
from exchange_websocket.utils import list_slice

from etc.register_monitor_msg import RegisterMonitorMsg # TEMP

In [2]:
with open("/home/kp_info_loader/kp_info_loader_config.json") as f:
    config = json.load(f)

node = "TEST"
monitor_bot_name = config['monitor_setting']['monitor_bot']
monitor_bot_token = config['telegram_bot_setting'][monitor_bot_name]
monitor_bot_api_url = config['monitor_setting']['monitor_bot_api_url']
admin_id_list = []
admin_id = config['telegram_admin_id']['charlie1155']
admin_id_list.append(admin_id)
logging_dir = None
register_monitor_msg = RegisterMonitorMsg(monitor_bot_token, monitor_bot_api_url, admin_id, logging_dir)

In [3]:
class BithumbWebsocket:
    def __init__(self, admin_id, node, proc_n, get_symbol_list, register_monitor_msg, logging_dir):
        self.admin_id = admin_id
        self.node = node
        self.url = "wss://pubwss.bithumb.com/pub/ws"
        self.register_monitor_msg = register_monitor_msg
        self.get_symbol_list = get_symbol_list
        self.websocket_logger = KimpBotLogger("bithumb_websocket", logging_dir).logger
        manager = Manager()
        self.ticker_dict = manager.dict()
        self.orderbook_dict = manager.dict()
        self.proc_n = proc_n
        self.before_symbols_list = self.get_symbol_list()
        self.sliced_symbols_list = list_slice(self.get_symbol_list(), self.proc_n)
        self.stop_restart_webscoket = False
        self.price_proc_event_list = []
        self.websocket_proc_dict = {}
        self.websocket_symbol_dict = {}
        self._start_websocket()
        self.monitor_shared_symbol_change_thread = Thread(target=self.monitor_shared_symbol_change, daemon=True)
        self.monitor_shared_symbol_change_thread.start()
        self.monitor_websocket_last_update_thread = Thread(target=self.monitor_websocket_last_update, daemon=True)
        self.monitor_websocket_last_update_thread.start()

    def __del__(self):
        self.terminate_websocket()

    def bithumb_websocket(self, result_dict, data, error_event):
        def on_message(ws, message):
            if error_event.is_set():
                ws.close()
                raise Exception("bithumb_websocket|error_event is set. closing websocket..")
            message_dict = json.loads(message)
            # print(message_dict) # TEST
            if 'content' in message_dict.keys():
                result_dict[message_dict['content']['symbol']] = {**message_dict['content'], "last_update": datetime.datetime.utcnow()}


        def on_error(ws, error):
            # print(f'bithumb_websocket on_error executed!')
            # print(error)
            self.websocket_logger.error(f'bithumb_websocket|bithumb_websocket on_error executed!\n Error: {error}')

        def on_close(ws, close_status_code, close_msg):
            # print(f"\n\n### closed ###\nclose_msg: {close_msg}\nclose_status_code: {close_status_code}")
            self.websocket_logger.info(f"bithumb_websocket|\n\n### closed ###\nclose_msg: {close_msg}\nclose_status_code: {close_status_code}")
            pass # TEST

        def on_open(ws):
            # print(f'bithumb_websocket started')
            self.websocket_logger.info(f'bithumb_websocket|bithumb_websocket started')
            ws.send(json.dumps(data))

        websocket.enableTrace(False)
        ws = websocket.WebSocketApp(self.url,
                                on_open=on_open,
                                on_message=on_message,
                                on_error=on_error,
                                on_close=on_close)
        ws.run_forever(ping_interval=30)

    def _start_websocket(self):
        def handle_price_procs():
            while True:
                try:
                    if self.stop_restart_webscoket is False:
                        for i in range(self.proc_n):
                            ticker_start_proc = False
                            ticker_restarted = False
                            if f"{i+1}th_ticker_proc" in self.websocket_proc_dict.keys() and not self.websocket_proc_dict[f"{i+1}th_ticker_proc"].is_alive():
                                ticker_start_proc = True
                                ticker_restarted = True
                                self.websocket_proc_dict[f"{i+1}th_ticker_proc"].terminate()
                                self.websocket_logger.info(f"bithumb_orderbook_ticker_websocket|{i+1}th bithumb_ticker_proc terminated.")
                            elif f"{i+1}th_ticker_proc" not in self.websocket_proc_dict.keys():
                                ticker_start_proc = True
                                self.websocket_logger.info(f"{i+1}th bithumb ticker websocket does not exist. starting..")
                                self.websocket_logger.info(f"bithumb_orderbook_ticker_websocket|{i+1}th bithumb_ticker_proc started.")
                            if ticker_start_proc is True:
                                error_event = Event()
                                self.price_proc_event_list.append(error_event)
                                self.websocket_symbol_dict[f"{i+1}th_ticker_symbol"] = self.sliced_symbols_list[i]
                                ticker_data = {"type": "ticker", "symbols": self.sliced_symbols_list[i], "tickTypes": ["24H"]}
                                ticker_proc = Process(target=self.bithumb_websocket, args=(self.ticker_dict, ticker_data, error_event), daemon=True)
                                self.websocket_proc_dict[f"{i+1}th_ticker_proc"] = ticker_proc
                                ticker_proc.start()
                                if ticker_restarted:
                                    content = f"restarted {i+1}th Bithumb ticker websocket.. alive state: {self.websocket_proc_dict[f'{i+1}th_ticker_proc'].is_alive()}"
                                    self.websocket_logger.info(f"bithumb_orderbook_ticker_websocket|{content}")
                                    self.register_monitor_msg.register(self.admin_id, self.node, 'monitor', 'bithumb ticker websocket restart', content, code=None, sent_switch=0, send_counts=1, remark=None)

                            time.sleep(0.5)
                            orderbook_start_proc = False
                            orderbook_restarted = False
                            if f"{i+1}th_orderbook_proc" in self.websocket_proc_dict.keys() and not self.websocket_proc_dict[f"{i+1}th_orderbook_proc"].is_alive():
                                orderbook_start_proc = True
                                orderbook_restarted = True
                                self.websocket_proc_dict[f"{i+1}th_orderbook_proc"].terminate()
                                self.websocket_logger.info(f"bithumb_orderbook_ticker_websocket|{i+1}th bithumb_orderbook_proc terminated.")
                            elif f"{i+1}th_orderbook_proc" not in self.websocket_proc_dict.keys():
                                orderbook_start_proc = True
                                self.websocket_logger.info(f"{i+1}th bithumb orderbook websocket does not exist. starting..")
                                self.websocket_logger.info(f"bithumb_orderbook_ticker_websocket|{i+1}th bithumb_orderbook_proc started.")
                            if orderbook_start_proc is True:
                                error_event = Event()
                                self.price_proc_event_list.append(error_event)
                                self.websocket_symbol_dict[f"{i+1}th_orderbook_symbol"] = self.sliced_symbols_list[i]
                                orderbook_data = {"type": "orderbooksnapshot", "symbols": self.sliced_symbols_list[i]}
                                orderbook_proc = Process(target=self.bithumb_websocket, args=(self.orderbook_dict, orderbook_data, error_event), daemon=True)
                                self.websocket_proc_dict[f"{i+1}th_orderbook_proc"] = orderbook_proc
                                orderbook_proc.start()
                                if orderbook_restarted:
                                    content = f"restarted {i+1}th bithumb orderbook websocket.. alive state: {self.websocket_proc_dict[f'{i+1}th_orderbook_proc'].is_alive()}"
                                    self.websocket_logger.info(f"bithumb_orderbook_ticker_websocket|{content}")
                                    self.register_monitor_msg.register(self.admin_id, self.node, 'monitor', 'bithumb orderbook websocket restart', content, code=None, sent_switch=0, send_counts=1, remark=None)
                            time.sleep(0.5)
                except Exception as e:
                    content = f"handle_price_procs|{traceback.format_exc()}"
                    self.websocket_logger.error(content)
                    self.register_monitor_msg.register(self.admin_id, self.node, 'error', f"handle_price_procs", content=content, code=None, sent_switch=0, send_counts=1, remark=None)
                    time.sleep(1)
                time.sleep(0.5)
        self.handle_price_procs_thread = Thread(target=handle_price_procs, daemon=True)
        self.handle_price_procs_thread.start()

    def terminate_websocket(self):
        self.stop_restart_webscoket = True
        time.sleep(0.5)
        for each_event in self.price_proc_event_list:
            each_event.set()
        self.websocket_logger.info(f"[BITHUMB SPOT]all websockets' event has been set")
        self.price_proc_event_list = []
        time.sleep(1)
        self.stop_restart_webscoket = False
    
    def check_status(self, print_result=False, include_text=False):
        if len(self.websocket_proc_dict) == 0:
            proc_status = False
            print_text = "[BITHUMB SPOT]bithumb websocket proc is not running."
            if print_result:
                print(print_text)
            if include_text:
                return (proc_status, print_text)
            return proc_status
        else:
            proc_status = all([x.is_alive() for x in self.websocket_proc_dict.values()])
            print_text = ""
            for key, value in self.websocket_proc_dict.items():
                print_text += f"[BITHUMB SPOT]{key} status: {value.is_alive()}\n"
            if print_result:
                print(print_text)
            if include_text:
                return (proc_status, print_text)
            return proc_status

    def monitor_shared_symbol_change(self, loop_time_secs=60):
        self.websocket_logger.info("[BITHUMB SPOT]started monitor_shared_symbol_change..")
        while True:
            time.sleep(loop_time_secs)
            try:
                restart_websockets = False
                new_symbols_list = self.get_symbol_list()
                
                if sorted(self.before_symbols_list) != sorted(new_symbols_list):
                    restart_websockets = True
                    deleted_spot_shared_symbol = [x for x in self.before_symbols_list if x not in new_symbols_list]
                    added_spot_shared_symbol = [x for x in new_symbols_list if x not in self.before_symbols_list]
                    content = f"monitor_shared_symbol_change|[BITHUMB SPOT]SPOT shared symbol changed. deleted: {deleted_spot_shared_symbol}, added: {added_spot_shared_symbol}"
                    self.websocket_logger.info(content)
                    self.register_monitor_msg.register(self.admin_id, self.node, 'monitor', 'monitor_shared_symbol_change', content, code=None, sent_switch=0, send_counts=1, remark=None)
                    for each_spot_shared_symbol in deleted_spot_shared_symbol:
                        # remove deleted symbol from bithumb_ticker_dict and bithumb_orderbook_dict
                        try:
                            del self.ticker_dict[each_spot_shared_symbol]
                        except Exception:
                            pass
                        try:
                            del self.orderbook_dict[each_spot_shared_symbol]
                        except Exception:
                            pass
                
                if restart_websockets is True:
                    # Set the newer values to before values
                    self.before_symbols_list = new_symbols_list
                    # Set sliced values too
                    self.sliced_symbols_list = list_slice(self.get_symbol_list(), self.proc_n)
                    # terminate websockets
                    self.terminate_websocket()
            except Exception as e:
                content = f"monitor_shared_symbol_change|{traceback.format_exc()}"
                self.websocket_logger.error(content)
                self.register_monitor_msg.register(self.admin_id, self.node, 'error', f"monitor_shared_symbol_change", content=content, code=None, sent_switch=0, send_counts=1, remark=None)

    def monitor_websocket_last_update(self, update_threshold_mins=10, loop_time_secs=15):
        time.sleep(10)
        self.websocket_logger.info(f"[BITHUMB SPOT]started monitor_websocket_last_update..")
        while True:
            time.sleep(loop_time_secs)
            try:
                ticker_df = pd.DataFrame(dict(self.ticker_dict)).T.reset_index()
                orderbook_df = pd.DataFrame(dict(self.orderbook_dict)).T.reset_index()
                for i in range(self.proc_n):
                    allocated_symbol_list = self.websocket_symbol_dict[f"{i+1}th_ticker_symbol"]
                    # check ticker dict's last_update
                    allocated_ticker_df = ticker_df[ticker_df['symbol'].isin(allocated_symbol_list)]
                    if len(allocated_ticker_df) == 0:
                        content = f"monitor_websocket_last_update|{i+1}th_ticker_proc has no ticker_dict data. Restarting websocket.."
                        self.websocket_logger.info(content)
                        self.register_monitor_msg.register(self.admin_id, self.node, 'monitor', 'monitor_websocket_last_update', content, code=None, sent_switch=0, send_counts=1, remark=None)
                        self.websocket_proc_dict[f"{i+1}th_ticker_proc"].terminate()
                        continue
                    ticker_last_update = allocated_ticker_df['last_update'].max()
                    # check orderbook dict's last_update
                    allocated_orderbook_df = orderbook_df[orderbook_df['symbol'].isin(allocated_symbol_list)]
                    if len(allocated_orderbook_df) == 0:
                        content = f"monitor_websocket_last_update|{i+1}th_orderbook_proc has no orderbook_dict data. Restarting websocket.."
                        self.websocket_logger.info(content)
                        self.register_monitor_msg.register(self.admin_id, self.node, 'monitor', 'monitor_websocket_last_update', content, code=None, sent_switch=0, send_counts=1, remark=None)
                        self.websocket_proc_dict[f"{i+1}th_orderbook_proc"].terminate()
                        continue
                    orderbook_last_update = allocated_orderbook_df['last_update'].max()
                    # If the last update is older than update_threshold_mins, restart websocket
                    if (datetime.datetime.utcnow() - ticker_last_update).total_seconds() / 60 > update_threshold_mins:
                        content = f"monitor_websocket_last_update|{i+1}th_ticker_proc last_update is older than {update_threshold_mins} mins. Restarting websocket.."
                        self.websocket_logger.info(content)
                        self.register_monitor_msg.register(self.admin_id, self.node, 'monitor', 'monitor_websocket_last_update', content, code=None, sent_switch=0, send_counts=1, remark=None)
                        self.websocket_proc_dict[f"{i+1}th_ticker_proc"].terminate()
                    if (datetime.datetime.utcnow() - orderbook_last_update).total_seconds() / 60 > update_threshold_mins:
                        content = f"monitor_websocket_last_update|{i+1}th_orderbook_proc last_update is older than {update_threshold_mins} mins. Restarting websocket.."
                        self.websocket_logger.info(content)
                        self.register_monitor_msg.register(self.admin_id, self.node, 'monitor', 'monitor_websocket_last_update', content, code=None, sent_switch=0, send_counts=1, remark=None)
                        self.websocket_proc_dict[f"{i+1}th_orderbook_proc"].terminate()
            except Exception as e:
                content = f"monitor_websocket_last_update|{traceback.format_exc()}"
                self.websocket_logger.error(content)
                self.register_monitor_msg.register(self.admin_id, self.node, 'error', f"monitor_websocket_last_update", content=content, code=None, sent_switch=0, send_counts=1, remark=None)

    def get_price_df(self):
        orderbook_df = pd.DataFrame(self.orderbook_dict.values())
        ticker_df = pd.DataFrame(self.ticker_dict.values())
        orderbook_df['best_ask'] = orderbook_df['asks'].apply(lambda x: x[0][0])
        orderbook_df['best_bid'] = orderbook_df['bids'].apply(lambda x: x[0][0])
        merged_df = orderbook_df.merge(ticker_df[['symbol','chgRate','value']], on='symbol', how='inner')
        merged_df[['base_asset', 'quote_asset']] = merged_df['symbol'].str.split('_', expand=True)
        merged_df = merged_df.rename(columns={'value':'atp24h', 'chgRate':'scr', 'best_ask':'ap', 'best_bid':'bp'})
        merged_df.loc[:, ['scr','atp24h','ap','bp']] = merged_df.loc[:, ['scr','atp24h','ap','bp']].astype(float)
        merged_df['tp'] = np.nan
        return merged_df


In [4]:
bithumb_websocket = BithumbWebsocket(admin_id, node, 1, lambda: ["XRP_KRW", "BTC_KRW", "XRP_BTC", "ETH_BTC"], register_monitor_msg, logging_dir)

2023-11-08 13:22:14,490 - bithumb_websocket - INFO - 1th bithumb ticker websocket does not exist. starting..
2023-11-08 13:22:14,492 - bithumb_websocket - INFO - [BITHUMB SPOT]started monitor_shared_symbol_change..
2023-11-08 13:22:14,492 - bithumb_websocket - INFO - bithumb_orderbook_ticker_websocket|1th bithumb_ticker_proc started.


2023-11-08 13:22:14,622 - bithumb_websocket - INFO - bithumb_websocket|bithumb_websocket started
2023-11-08 13:22:15,004 - bithumb_websocket - INFO - 1th bithumb orderbook websocket does not exist. starting..
2023-11-08 13:22:15,005 - bithumb_websocket - INFO - bithumb_orderbook_ticker_websocket|1th bithumb_orderbook_proc started.
2023-11-08 13:22:15,048 - bithumb_websocket - INFO - bithumb_websocket|bithumb_websocket started


In [5]:
pd.DataFrame(bithumb_websocket.ticker_dict.values())

,volumePower,chgAmt,chgRate,prevClosePrice,buyVolume,sellVolume,volume,value,highPrice,lowPrice,closePrice,openPrice,time,date,tickType,symbol,last_update
0,106.87,-2.9,-0.32,911.4,115131131.08971175,107725644.26179852,222855789.76546027,203327421240.305842024,937.5,881,913.3,916.2,132215,20231108,24H,XRP_KRW,2023-11-08 04:22:16.286982
1,134.26,476000,1.02,46600000,2222.12385253,1655.03636661,3877.16021914,181512608302.06099,47680000,46250000,47080000,46604000,132214,20231108,24H,BTC_KRW,2023-11-08 04:22:16.187509


In [7]:
price_df = bithumb_websocket.get_price_df()
price_df

,symbol,datetime,asks,bids,last_update,ap,bp,scr,atp24h,base_asset,quote_asset,tp
0,BTC_KRW,1699417347981352,"[[47076000, 0.0199], [47079000, 0.0238], [4708...","[[47074000, 0.0114], [47072000, 0.0491], [4707...",2023-11-08 04:22:28.093759,47076000.0,47074000.0,1.01,181516511750.035492,BTC,KRW,NaN
1,XRP_KRW,1699417347671068,"[[913.5, 517.769], [913.6, 809.9825], [913.7, ...","[[913.2, 2816.3021], [913.1, 29374.7826], [913...",2023-11-08 04:22:27.758511,913.5,913.2,-0.33,203344910619.467316,XRP,KRW,NaN


In [14]:
ticker_df = pd.DataFrame(bithumb_websocket.ticker_dict.values())
ticker_df


,volumePower,chgAmt,chgRate,prevClosePrice,buyVolume,sellVolume,volume,value,highPrice,lowPrice,closePrice,openPrice,time,date,tickType,symbol,last_update
0,134.24,321000,0.69,46600000,2223.73812488,1656.55074447,3880.28886935,181631965487.61059,47680000,46250000,46982000,46661000,123459,20231108,24H,BTC_KRW,2023-11-08 03:35:02.993948
1,106.61,-6.8,-0.74,911.4,115752556.11576048,108577409.14007811,224328979.66978859,204695261956.352062963,937.5,881,911.6,918.4,123504,20231108,24H,XRP_KRW,2023-11-08 03:35:04.057909
2,164.7,-0.00000048,-2.43,0.00001935,67133.317484891674589644,40761.1384,107894.455884891674589644,2.117043715604,0.00002049,0.00001894,0.00001931,0.00001979,123359,20231108,24H,XRP_BTC,2023-11-08 03:34:00.028894


,symbol,datetime,asks,bids,last_update,ap,bp,scr,atp24h,base_asset,quote_asset
0,XRP_KRW,1699414505369125,"[[911.7, 5942.93], [911.8, 3239.4617], [911.9,...","[[911.6, 353.527], [911.5, 18342.5467], [911.4...",2023-11-08 03:35:05.421736,911.7,911.6,-0.74,204695261956.352051,XRP,KRW
1,BTC_KRW,1699414505978362,"[[46983000, 0.0913], [46984000, 0.023], [46985...","[[46982000, 1.7452], [46981000, 0.0211], [4698...",2023-11-08 03:35:06.019108,46983000.0,46982000.0,0.69,181631965487.610596,BTC,KRW
2,XRP_BTC,1699414503875690,"[[0.00002012, 156.5781], [0.00002013, 23184.80...","[[0.00001931, 65559.3427], [0.00001892, 7037.2...",2023-11-08 03:35:03.890546,0.00002,0.000019,-2.43,2.117044,XRP,BTC


In [32]:
error_event = Event()
result_dict = {}
data = {
    "type": "orderbooksnapshot",
    "symbols": ["XRP_KRW"],
    # "tickTypes": ["MID"],
}
bithumb_websocket(result_dict, data, error_event)

bithumb_websocket started
{'status': '0000', 'resmsg': 'Connected Successfully'}
{'status': '0000', 'resmsg': 'Filter Registered Successfully'}
{'type': 'orderbooksnapshot', 'content': {'symbol': 'XRP_KRW', 'datetime': '1699342138938631', 'asks': [['922.8', '7008.4997'], ['922.9', '24520.1559'], ['923', '11106.2695'], ['923.1', '1741.0642'], ['923.2', '8280.7396'], ['923.3', '50024.8761'], ['923.4', '31250.0711'], ['923.5', '5405.5925'], ['923.8', '362.6134'], ['923.9', '46651.0264'], ['924', '237.4906'], ['924.1', '2056.54'], ['924.4', '5109.2239'], ['924.6', '8886.518'], ['924.7', '2144.1146'], ['924.8', '16606.0351'], ['924.9', '11432.737'], ['925', '22722.217'], ['925.1', '5148.8531'], ['925.3', '17.4209'], ['925.5', '47.4605'], ['925.8', '20436.711'], ['926', '20601.4048'], ['926.2', '37309.6297'], ['926.3', '11434.2334'], ['926.4', '4000'], ['926.7', '10790.9788'], ['926.8', '52813'], ['926.9', '38868.3705'], ['927', '36346']], 'bids': [['922.7', '280'], ['922.6', '988.4036'], ['